# Lab 5: Converting a PyTorch Model to ONNX and Running Inference Using ONNX Runtime

### Objective:
Train a basic PyTorch model, convert it to ONNX format, and perform inference using ONNX Runtime.

### Pre-requisites:
- PyTorch
- ONNX and ONNX Runtime installed (`pip install onnx onnxruntime`)
- Basic understanding of neural networks and PyTorch


In [7]:
# Step 1: Import Required Libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import onnx
import onnxruntime as ort
import numpy as np

In [8]:
# Step 2: Define a Simple Neural Network
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(28*28, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [9]:
# Step 3: Load Data and Train the Model
transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

model = SimpleNet()
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(1):
    for data, target in train_loader:
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
print("Training completed.")

Training completed.


In [10]:
# Step 4: Export the Model to ONNX Format
dummy_input = torch.randn(1, 1, 28, 28)
torch.onnx.export(model, dummy_input, "simple_mnist.onnx", input_names=['input'], output_names=['output'],
                  dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
                  opset_version=11)
print("Model exported to simple_mnist.onnx")

Model exported to simple_mnist.onnx


In [11]:
# Step 5: Load ONNX Model and Run Inference Using ONNX Runtime
onnx_model = onnx.load("simple_mnist.onnx")
onnx.checker.check_model(onnx_model)

ort_session = ort.InferenceSession("simple_mnist.onnx")

# Sample image from test set
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
x_test, y_test = test_dataset[0]

# Convert to numpy format
x_test = x_test.unsqueeze(0).numpy().astype(np.float32)

# Run inference
outputs = ort_session.run(None, {'input': x_test})
predicted = np.argmax(outputs[0])
print(f"Predicted Label: {predicted}, Ground Truth: {y_test}")

Predicted Label: 7, Ground Truth: 7
